# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Loaded dataset: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This information will help identify what is available for extraction in the FAIR^2 dataset.

In [ ]:
# List all available record sets and their fields, referencing by @id
print("Available record sets (by @id):")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    print("  Fields:")
    for field in rs.get('fields', []):
        print(f"    - Field @id: {field['@id']}, Name: {field.get('name', '(no name)')}, DataType: {field.get('dataType', '(unknown)')}")
    print()# For further exploration, pick the first record set if available
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    example_fields = record_sets[0].get('fields', [])
else:
    example_record_set_id = None
    example_fields = []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If there are record sets, extract data for each using their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []
# Show the steps to extract data from each record set
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet @id: {record_set_id} with {df.shape[0]} rows and {df.shape[1]} columns.")
            print(f"First 5 columns: {df.columns[:5].tolist()}")
        else:
            print(f"RecordSet @id: {record_set_id} is defined, but contains no records.")
    except Exception as e:
        print(f"Failed to load records for RecordSet @id: {record_set_id}. Error: {e}")# For demonstration, select first record set (if present) to continue EDA
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    df = dataframes[selected_record_set_id] if selected_record_set_id in dataframes else None
    print(f"Available columns in selected DataFrame (@id={selected_record_set_id}):\n", df.columns.tolist() if df is not None else 'NONE')
else:
    selected_record_set_id = None
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—filtering, normalization, grouping—using field `@id`s.

*Remove outliers, normalize numeric fields, and group/categorize data as relevant.*

In [ ]:
# The following example assumes one or more numeric fields present in the DataFrame.
# We'll select the first float or integer typed field for demonstration.
import numpy as np

if df is not None and not df.empty:
    print(f"Columns in selected DataFrame: {df.columns.tolist()}")

    # Try to find a numeric field by dtype
    numeric_candidate = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidate = col
            break
    if numeric_candidate is None:
        # Try to parse columns which look numeric (object columns but should be floats/ints)
        for col in df.columns:
            try:
                converted = pd.to_numeric(df[col], errors='coerce')
                if converted.notna().sum() > 0:
                    numeric_candidate = col
                    df[col] = converted
                    break
            except Exception:
                continue
    if numeric_candidate is not None:
        print(f"Using numeric field for analysis: {numeric_candidate}")
        threshold = df[numeric_candidate].mean() if df[numeric_candidate].notna().sum() else 10

        # Filter for records above mean value as an example
        filtered_df = df[df[numeric_candidate] > threshold]
        print(f"Filtered records with {numeric_candidate} > {threshold:.2f} ({filtered_df.shape[0]} records):")
        display(filtered_df.head())

        # Normalize
        normed_column = numeric_candidate + "_normalized"
        filtered_df[normed_column] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
        print(f"Normalized {numeric_candidate} for filtered records:")
        display(filtered_df[[numeric_candidate, normed_column]].head())
        # Try grouping by a non-numeric field (categorical/text/etc)
        group_field = None
        for col in df.columns:
            if col != numeric_candidate and df[col].nunique() < df.shape[0] // 2:
                group_field = col
                break
        if group_field is not None:
            print(f"Grouping filtered records by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().reset_index()
            display(grouped_df.head())
    else:
        print("No numeric field found for demonstration.")
else:
    print("No records available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here's an example plot of the distribution of the numeric field used above (if available):

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_candidate is not None and numeric_candidate in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_candidate].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_candidate}")
    plt.xlabel(numeric_candidate)
    plt.ylabel("Count")
    plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset package using the `mlcroissant` library. We loaded record sets using their `@id` references, examined available fields, and performed basic exploratory data analysis (EDA) and visualization on numeric columns detected in the dataset.

Further analyses may include domain-specific investigations, more advanced feature engineering, or integration with external bioinformatics tools depending on the research question.